# 方案一：数据预处理与探索性分析（EDA）复现

本 Notebook 从 `data/raw/` 的原始 CSV 开始，完整复现项目的第一阶段：

1. **文本清洗**：HTML 实体解码、情感表情转中文、去 URL / @ / #话题#、品牌脱敏符 `**` 归一化为"某品牌"、只保留中英文数字；
2. **jieba 分词**：把 `Train_labels.csv` 中的标注词与化妆品领域词注入用户词典，并谨慎去停用词（保留否定词和程度副词）；
3. **标签解析**：`_` 占位符转 NA，字符偏移转可空整数 `Int64`；
4. **EDA**：数据规模、长度 / 每条评论四元组数 / 类别 / 极性分布、缺失重复异常检查、分词质量验证，并输出 5 张图。

> 本方案不训练神经网络，纯 CPU 可跑；脚本产物路径与配置文件 `src/config.py` 保持一致，全部写入项目根目录下的 `outputs/`。

## Step 0. 环境与依赖

首次运行请取消注释安装依赖（**国内机器先配清华镜像可加速 5~10 倍**）：

```bash
# 一次性配置清华镜像（永久生效）
pip config set global.index-url https://pypi.tuna.tsinghua.edu.cn/simple
pip config set global.trusted-host pypi.tuna.tsinghua.edu.cn

# 主依赖（本 Notebook 只用 pandas / numpy / jieba / matplotlib / PyYAML）
pip install -r requirements.txt
```

EDA 画图使用 matplotlib；Windows 下中文显示依赖 SimHei / Microsoft YaHei 字体。

In [ ]:
# 首次运行可取消注释（已安装则跳过）
# %pip install -q -r requirements.txt

import importlib.util
need = {"pandas": "pandas", "numpy": "numpy", "jieba": "jieba",
        "matplotlib": "matplotlib", "yaml": "yaml"}
missing = [pkg for pkg, mod in need.items() if importlib.util.find_spec(mod) is None]
print("依赖检查:", "全部就绪" if not missing else f"缺少 {missing}，请先运行 pip -r requirements.txt")

In [ ]:
# 统一入口：定位项目根目录（notebook/ 的上一级），把 src/ 加入 import 路径
# 脚本产物路径保持原样（config.py 中的 BASE_DIR/outputs, BASE_DIR/data/...）
import os, sys, subprocess
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 脚本产物写入的根目录（与 src/config.py 保持一致）
ARTIFACT_DIR = PROJECT_ROOT

# 默认使用 0 号 GPU；多卡机器可改成 "1" 等
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

print("项目根目录:", PROJECT_ROOT)
print("产物根目录:", ARTIFACT_DIR)

def run(cmd):
    """用当前 Python 解释器执行命令，工作目录为 PROJECT_ROOT。"""
    print(">>", " ".join(map(str, cmd)))
    subprocess.run([str(c) for c in cmd], check=True, env=os.environ.copy())

## Step 1. 原始数据校验

原始数据需保持官方 CSV 原样放在 `data/raw/`：

```
data/raw/TRAIN/Train_reviews.csv   # id, Reviews
data/raw/TRAIN/Train_labels.csv    # id, AspectTerms, A_start, A_end, OpinionTerms, O_start, O_end, Categories, Polarities
data/raw/TEST/Test_reviews.csv     # id, Reviews
```

In [ ]:
import pandas as pd
from pathlib import Path

paths = [
    "data/raw/TRAIN/Train_reviews.csv",
    "data/raw/TRAIN/Train_labels.csv",
    "data/raw/TEST/Test_reviews.csv",
]
for p in paths:
    print(f"{p:45s} 存在: {Path(p).exists()}")

train = pd.read_csv("data/raw/TRAIN/Train_reviews.csv", encoding="utf-8")
labels = pd.read_csv("data/raw/TRAIN/Train_labels.csv", encoding="utf-8")
test = pd.read_csv("data/raw/TEST/Test_reviews.csv", encoding="utf-8")
print("训练评论:", train.shape, "| 训练标签:", labels.shape, "| 测试评论:", test.shape)
assert len(train) == 3229 and len(labels) == 6633 and len(test) == 2237, "数据规模与预期不一致，请检查原始数据"
display(train.head(3))

## Step 2. 观察清洗与分词逻辑

`src/preprocess.py` 的 `clean_text` / `tokenize` / `filter_tokens` 是后续所有方案的公共底座。下面直接调用这几个函数观察效果。

In [ ]:
from preprocess import clean_text, tokenize, filter_tokens

samples = [
    "物流很快👍 https://t.cn/xx @客服 #试用# **品牌**，但是遮瑕效果一般！",
    "一般",
]
for s in samples:
    cleaned = clean_text(s)
    print("原文:", s)
    print("清洗:", cleaned)
    print("分词:", filter_tokens(tokenize(cleaned)))
    print()

## Step 3. 运行预处理脚本

```bash
python src/preprocess.py
```

产物：
- `outputs/processed/Train_reviews_processed.csv`、`Test_reviews_processed.csv`（id, Reviews, clean_text, tokens）
- `outputs/processed/Train_labels_parsed.csv`（结构化标签）
- `outputs/logs/preprocess_report.txt`

In [ ]:
run([sys.executable, "src/preprocess.py"])

In [ ]:
print((ARTIFACT_DIR / "outputs/logs/preprocess_report.txt").read_text(encoding="utf-8"))

In [ ]:
train_proc = pd.read_csv(ARTIFACT_DIR / "outputs/processed/Train_reviews_processed.csv")
test_proc = pd.read_csv(ARTIFACT_DIR / "outputs/processed/Test_reviews_processed.csv")
labels_parsed = pd.read_csv(ARTIFACT_DIR / "outputs/processed/Train_labels_parsed.csv")

print("训练预处理:", train_proc.shape, "| 测试预处理:", test_proc.shape)
print("标签解析:", labels_parsed.shape)
print("\n标签列类型:")
print(labels_parsed.dtypes)
display(train_proc.head(3))
display(labels_parsed.head(5))

## Step 4. 运行 EDA

```bash
python src/eda.py
```

EDA 会复用 `preprocess.py` 的清洗/分词逻辑，检查数据规模、分布、缺失值、重复值、位置与术语一致性，并生成 5 张图。

In [ ]:
run([sys.executable, "src/eda.py"])

In [ ]:
text = (ARTIFACT_DIR / "outputs/logs/eda_report.txt").read_text(encoding="utf-8")
print(text)

## Step 5. 查看 EDA 图表

In [ ]:
from IPython.display import Image, display
figs = sorted((ARTIFACT_DIR / "outputs/figures").glob("*.png"))
print("图表数:", len(figs))
for p in figs:
    print(p.name)
    display(Image(filename=str(p), width=760))

## 关键结论（直接影响后续建模）

1. **输出数量不固定**：37.4% 的评论含 1 个四元组，34.5% 含 2 个，最多 7 个，因此不能用普通单标签分类；
2. **隐式方面占 71.4%**：只抽取显式方面词最多覆盖不到 30% 的场景，必须显式建模“无方面词”；
3. **类别与极性极度不均衡**：整体 2822 个 vs 新鲜度 13 个；正面约占 89%；
4. **标签质量高**：无缺失、无越界取值，术语与位置缺失完全一致。

这些结论分别对应后续 Baseline 的隐式方面门控、BERT 的 `[CLS]` 分类头、管道 Stage2 的类别权重，以及融合策略的设计动机。